# Warehouse Throughput Analysis


## Project Overview

This notebook analyses daily warehouse operations data to study throughput, efficiency by shift, error rates, and staff productivity. The goal is to surface operational bottlenecks and guide staffing and process decisions.

## Learning Objectives

- Measure and compare throughput across warehouses and shifts
- Identify productivity and error rate drivers
- Detect seasonal or trend patterns in daily operations
- Apply statistical comparisons between operational groups
- Translate findings into operational recommendations

## Business or Research Problem

Warehouse managers need to know: **which shifts, warehouses, and time periods are underperforming** — and why — so they can improve throughput without increasing headcount.

## Why This Analysis Matters

Warehouse throughput directly affects order fulfillment speed and customer satisfaction. Identifying and fixing bottlenecks can reduce cost-per-unit processed and improve SLA compliance without capital investment.

## Dataset Overview

| Field | Description |
|-------|-------------|
| date | Operation date |
| warehouse_id | Warehouse identifier (WH-1 to WH-4) |
| inbound_units | Units received |
| outbound_units | Units shipped |
| returns_processed | Return units handled |
| staff_count | Number of staff on shift |
| shift | Morning / Afternoon / Night |
| errors | Mis-picks, mis-ships, damage events |
| processing_time_min | Average unit processing time |

730 days × 3 shifts × 4 warehouses = 8,760 rows.

## Dataset Source and License Notes

Fully synthetic dataset generated for educational purposes. No real warehouse or company data is used.

## Environment Setup


In [ ]:
import importlib
for lib in ['numpy','pandas','matplotlib','seaborn','scipy']:
    print(f'{lib}: {importlib.import_module(lib).__version__}')

## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## Configuration / Constants


In [ ]:
SEED = 42
np.random.seed(SEED)
WAREHOUSES = ['WH-1','WH-2','WH-3','WH-4']
SHIFTS = ['Morning','Afternoon','Night']
FIG_W, FIG_H = 12, 5

## Dataset Download or Loading

We generate 2 years of daily warehouse operation data across 4 warehouses and 3 shifts.

In [ ]:
dates = pd.date_range('2022-01-01', periods=730)
rows = []
for d in dates:
    for wh in WAREHOUSES:
        for shift in SHIFTS:
            wh_factor = {'WH-1':1.0,'WH-2':0.85,'WH-3':1.15,'WH-4':0.95}[wh]
            shift_factor = {'Morning':1.2,'Afternoon':1.0,'Night':0.75}[shift]
            staff = int(np.random.normal(15*wh_factor*shift_factor, 2))
            staff = max(5, staff)
            inbound = int(np.random.normal(300*wh_factor*shift_factor, 40))
            outbound = int(np.random.normal(280*wh_factor*shift_factor, 35))
            returns = int(np.random.normal(30, 8))
            errors = int(np.random.poisson(2.5 / wh_factor))
            proc_time = round(np.random.normal(4.5 / shift_factor, 0.8), 1)
            rows.append([d, wh, shift, max(0,inbound), max(0,outbound),
                         max(0,returns), max(5,staff), max(0,errors), max(0.5,proc_time)])

df = pd.DataFrame(rows, columns=['date','warehouse_id','shift','inbound_units',
                                   'outbound_units','returns_processed','staff_count',
                                   'errors','processing_time_min'])
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week.astype(int)
df['year'] = df['date'].dt.year
df['total_units'] = df['inbound_units'] + df['outbound_units'] + df['returns_processed']
df['units_per_staff'] = (df['total_units'] / df['staff_count']).round(1)
df['error_rate'] = (df['errors'] / df['total_units'] * 100).round(3)

print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks


In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nUnits range: {df.total_units.min()} – {df.total_units.max()}')
print(f'Staff range: {df.staff_count.min()} – {df.staff_count.max()}')
print(f'Error rate range: {df.error_rate.min():.3f}% – {df.error_rate.max():.3f}%')

## Data Cleaning


In [ ]:
df = df[df['total_units'] > 0].copy()
df = df[df['staff_count'] > 0].copy()
print(f'Clean dataset: {len(df):,} rows')

## Exploratory Data Analysis

High-level throughput summary by warehouse and shift.

In [ ]:
summary = df.groupby(['warehouse_id','shift']).agg(
    avg_outbound=('outbound_units','mean'),
    avg_staff=('staff_count','mean'),
    avg_uph=('units_per_staff','mean'),
    avg_error_rate=('error_rate','mean')
).round(2)
print(summary)

## Univariate Analysis

We examine the distribution of daily total units processed and error rates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))
axes[0].hist(df['total_units'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Total Units Processed')
axes[0].set_title('Distribution of Daily Throughput')
axes[1].hist(df['error_rate'], bins=40, color='tomato', edgecolor='white')
axes[1].set_xlabel('Error Rate (%)')
axes[1].set_title('Distribution of Error Rate')
plt.tight_layout()
plt.show()

Throughput is roughly normally distributed. Error rates are right-skewed — most sessions have low error rates but a tail of high-error sessions drives average error rates up.

## Bivariate / Multivariate Analysis


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(FIG_W, 10))

# Throughput by warehouse
df.boxplot(column='total_units', by='warehouse_id', ax=axes[0,0])
axes[0,0].set_title('Throughput by Warehouse')
axes[0,0].set_xlabel('Warehouse')

# Throughput by shift
df.boxplot(column='total_units', by='shift', ax=axes[0,1])
axes[0,1].set_title('Throughput by Shift')
axes[0,1].set_xlabel('Shift')

# Units per staff by warehouse
df.boxplot(column='units_per_staff', by='warehouse_id', ax=axes[1,0])
axes[1,0].set_title('Units per Staff by Warehouse')
axes[1,0].set_xlabel('Warehouse')

# Error rate by warehouse
df.groupby('warehouse_id')['error_rate'].mean().plot(kind='bar', ax=axes[1,1], color='coral')
axes[1,1].set_title('Average Error Rate by Warehouse')
axes[1,1].set_ylabel('Error Rate (%)')

for ax in axes.flat:
    ax.title.set_size(10)
plt.suptitle('')
plt.tight_layout()
plt.show()

WH-3 leads in throughput while WH-2 lags. Night shifts process the fewest units but may have higher error rates due to fatigue. WH-1 has the lowest error rate.

## Feature-Specific Insights

Productivity (units per staff) vs error rate relationship.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
for wh, grp in df.groupby('warehouse_id'):
    ax.scatter(grp['units_per_staff'], grp['error_rate'], alpha=0.3, label=wh, s=15)
ax.set_xlabel('Units per Staff')
ax.set_ylabel('Error Rate (%)')
ax.set_title('Productivity vs Error Rate by Warehouse')
ax.legend()
plt.tight_layout()
plt.show()

## Statistical Checks

ANOVA test: is there a significant difference in throughput across warehouses?

In [ ]:
groups = [df[df['warehouse_id']==wh]['total_units'].values for wh in WAREHOUSES]
f_stat, p_val = stats.f_oneway(*groups)
print(f'One-way ANOVA — Total Units by Warehouse')
print(f'  F = {f_stat:.2f}, p = {p_val:.6f}')
if p_val < 0.05:
    print('  Warehouse differences are statistically significant.')

## Visual Analysis

Monthly throughput trend to detect seasonality or operational drift.

In [ ]:
monthly_throughput = df.groupby(['year','month'])['total_units'].mean().reset_index()
monthly_throughput['period'] = monthly_throughput['year'].astype(str) + '-' + monthly_throughput['month'].astype(str).str.zfill(2)

plt.figure(figsize=(FIG_W, FIG_H))
plt.plot(monthly_throughput['period'], monthly_throughput['total_units'], marker='o', color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title('Monthly Average Daily Throughput')
plt.ylabel('Avg Units Processed')
plt.tight_layout()
plt.show()

## Key Findings


In [ ]:
best_wh = df.groupby('warehouse_id')['total_units'].mean().idxmax()
best_shift = df.groupby('shift')['total_units'].mean().idxmax()
highest_err = df.groupby('warehouse_id')['error_rate'].mean().idxmax()
print(f'Best throughput warehouse: {best_wh}')
print(f'Best throughput shift: {best_shift}')
print(f'Highest error rate warehouse: {highest_err}')
print(f'Overall avg throughput: {df["total_units"].mean():.0f} units/day')
print(f'Overall error rate: {df["error_rate"].mean():.3f}%')

## Limitations

- No individual worker performance data
- Processing time is averaged — does not capture intra-shift variation
- Returns are not broken down by reason or handling complexity

## Recommendations / Next Steps

1. Share WH-3 best practices with WH-2 — the throughput gap suggests process differences worth investigating.
2. Review Night shift protocols to reduce error rates.
3. Consider cross-training to improve Morning shift flexibility during demand peaks.
4. Build a real-time throughput dashboard for shift supervisors.

## Common Mistakes

- Comparing raw unit counts without normalising by staff count
- Ignoring shift length differences when computing productivity
- Treating all error types the same — a mis-pick and a damaged item have different cost implications

## Mini Challenge / Exercises

1. Compute the correlation matrix between throughput, staff count, errors, and processing time.
2. Identify the top 10 single-day sessions with the highest error rate and check if they share a common warehouse or shift.
3. Plot a heatmap of average throughput by warehouse and month.

## Final Summary / Key Takeaways

- WH-3 leads in throughput; WH-2 is the laggard requiring process review.
- Morning shifts outperform Night shifts significantly — fatigue and staffing are likely factors.
- Error rates vary meaningfully by warehouse, pointing to process and training differences.
- ANOVA confirms that warehouse throughput differences are statistically significant, not random.
- The next step is benchmarking WH-3 practices and piloting them in WH-2.